# Lesson 0.2: Types of Data

Build a small student table with a nominal, an ordinal, a discrete and a continuous column, and see which summaries make sense for each.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A school office keeps a table of ten students with their eye colour, grade level, number of siblings and height. A new assistant wants to average every column. Which columns can we sensibly average, and what should we report for the others?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Build a table with four kinds of data

One column per type: eye colour (nominal), grade level (ordinal, so we tell pandas the order), siblings (discrete, counted) and height (continuous, measured). The dtypes show how pandas stores each one.

In [1]:
import pandas as pd  # Load pandas, Python's tool for tables of data, under the short name pd.

df = pd.DataFrame({  # Start building a DataFrame (a table). Each entry below is one column, written as name: list of values.
    'eye': ['brown','blue','brown','green','brown','blue','hazel','brown','green','blue'],  # Column eye: ten eye colours. They are names with no order, so the type is nominal.
    'grade': ['Sophomore','Senior','Freshman','Junior','Sophomore','Senior','Freshman','Sophomore','Junior','Senior'],  # Column grade: the school year. The years are ranked but the gaps are not equal, so the type is ordinal.
    'siblings': [1, 0, 2, 3, 1, 1, 0, 2, 4, 1],  # Column siblings: a count of brothers and sisters, so a discrete number.
    'height_cm': [172.4, 165.1, 180.3, 158.9, 175.6, 169.8, 162.2, 177.0, 171.3, 183.5],  # Column height_cm: a measurement, where any value in a range is possible, so continuous.
})  # Close the dictionary and the DataFrame call. Every opening bracket needs its partner.
df['eye'] = df['eye'].astype('category')  # Tell pandas that eye is a category. It then stops treating colours as free text and refuses arithmetic on them.
df['grade'] = pd.Categorical(df['grade'],  # Turn grade into a category too, and start listing its order, because the order is what makes it ordinal.
    categories=['Freshman','Sophomore','Junior','Senior'], ordered=True)  # Give the order Freshman to Senior and set ordered=True, so pandas can compare grades with each other.
print(df.dtypes)  # Print the dtypes, a quick check that each column is stored as the kind of data we intended.

eye          category
grade        category
siblings        int64
height_cm     float64
dtype: object


**What this means.** eye and grade are stored as categories, while siblings is a whole-number count (int64) and height_cm a measured decimal (float64). That is the first split of the lesson: two columns hold categories and two hold amounts. The next steps show why that matters.

## Step 2: Count the categories

For categories the natural summary is a count (or a percentage), not an average. value_counts does it. With sort=False the ordinal column stays in its natural order, from Freshman to Senior.

In [2]:
print(df['eye'].value_counts())  # Count how many students have each eye colour. Counting is the natural summary for names.
print()  # Print an empty line to separate the tables.
print(df['grade'].value_counts(sort=False))  # Count each grade. sort=False keeps the natural order Freshman to Senior instead of ordering by size.
print()  # Print another empty line.
print('share Junior or above:', (df['grade'] >= 'Junior').mean())  # Compare each grade with Junior (allowed because grade is ordered). Taking the mean of True and False gives the share.

eye
brown    4
blue     3
green    2
hazel    1
Name: count, dtype: int64

grade
Freshman     2
Sophomore    3
Junior       2
Senior       3
Name: count, dtype: int64

share Junior or above: 0.5


**What this means.** Brown is the most common eye colour (4 of 10), then blue (3), green (2) and hazel (1). The grades split into 2 Freshman, 3 Sophomore, 2 Junior and 3 Senior, and half the students (0.5) are Junior or above. For categories we report counts and shares, never an average.

## Step 3: Describe the numbers

Counts and measurements are real amounts, so mean, standard deviation and quartiles all make sense. describe gives them in one table.

In [3]:
print(df[['siblings', 'height_cm']].describe().round(2))  # describe() gives the count, mean, standard deviation, minimum, quartiles and maximum of numerical columns. round(2) keeps it readable.

       siblings  height_cm
count     10.00      10.00
mean       1.50     171.61
std        1.27       7.88
min        0.00     158.90
25%        1.00     166.28
50%        1.00     171.85
75%        2.00     176.65
max        4.00     183.50


**What this means.** Students have 1.50 siblings on average (from 0 to 4, median 1) and an average height of 171.61 cm with a standard deviation of 7.88 cm. These summaries make sense because the values are real amounts: one more sibling or one more centimetre always means the same step.

## Step 4: Why averaging a category makes no sense

Pandas refuses to average a category. If you force it by turning grades into codes 1 to 4, you get a number like 2.6 that is not a grade, and it pretends the gaps between grades are equal. For an ordinal column use counts, the maximum or a share instead.

In [4]:
for col in ['eye', 'grade']:  # Loop over the two category columns, to try the same operation on each of them.
    try:  # try: attempt something that might fail, so that an error is caught and does not stop the whole program.
        df[col].mean()  # Ask for the column's mean. For categories pandas refuses, which is its way of saying the answer would be meaningless.
    except TypeError:  # except TypeError: runs only if the attempt failed with that kind of error.
        print(f'mean of {col}: not allowed (TypeError)')  # Report the refusal, putting the column name into the message.

codes = df['grade'].cat.codes + 1  # Force it anyway: turn grades into codes 0 to 3, then add 1 to get 1 to 4.
print('mean of grade codes:', codes.mean(), '(is that a grade? no)')  # Average the codes. It works as arithmetic, but the answer is not a grade and it pretends all the gaps are equal.
print('highest grade present:', df['grade'].max())  # For an ordinal column a sensible summary is the highest category present.

mean of eye: not allowed (TypeError)
mean of grade: not allowed (TypeError)
mean of grade codes: 2.6 (is that a grade? no)
highest grade present: Senior


**What this means.** Pandas refuses to average eye and grade. Forcing the grades into codes gives 2.6, which is not a grade and only means something if every step between grades were equal. Report counts or the highest grade present (Senior) instead.

## So, what is the answer?

Only siblings and height_cm can be sensibly averaged: 1.5 siblings and 171.6 cm. Eye colour (nominal) should be counted, and grade (ordinal) is best reported with counts, shares or the highest grade, since an average of 2.6 is not a grade. Caution: a column full of digits is not automatically an amount, so always ask what one step of one unit means.

## Now you try

A librarian records 12 members: their membership type (student, staff or public), a satisfaction rating (Poor, OK, Good, Great), the books they borrowed last month and the minutes they spent per visit. She asks which summary suits each column.

**Data:** member = ['student','staff','public','student','student','public','staff','student','public','student','staff','public']; rating = ['Good','Great','OK','Good','Poor','Good','Great','OK','Good','Great','Good','OK']; books = [3,1,0,4,2,1,5,2,0,3,1,2]; minutes = [42.5,25.0,18.2,55.1,31.4,22.8,38.0,47.3,15.6,60.2,29.9,33.3]

1. Name the type of each of the four columns (nominal, ordinal, discrete or continuous).
2. Count the members per membership type and per rating, keeping ratings in order.
3. Find the share who rated Good or Great.
4. Describe the books and minutes columns.

*Hint:* Make a DataFrame, turn rating into an ordered Categorical, use value_counts for categories and describe() for the two numerical columns.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import pandas as pd

lib = pd.DataFrame({
    'member': ['student','staff','public','student','student','public','staff','student','public','student','staff','public'],
    'rating': ['Good','Great','OK','Good','Poor','Good','Great','OK','Good','Great','Good','OK'],
    'books': [3, 1, 0, 4, 2, 1, 5, 2, 0, 3, 1, 2],
    'minutes': [42.5, 25.0, 18.2, 55.1, 31.4, 22.8, 38.0, 47.3, 15.6, 60.2, 29.9, 33.3],
})
lib['rating'] = pd.Categorical(lib['rating'], categories=['Poor','OK','Good','Great'], ordered=True)
print(lib['member'].value_counts())
print(lib['rating'].value_counts(sort=False))
print('share Good or Great:', round((lib['rating'] >= 'Good').mean(), 2))
print(lib[['books', 'minutes']].describe().round(2))

member
student    5
public     4
staff      3
Name: count, dtype: int64
rating
Poor     1
OK       3
Good     5
Great    3
Name: count, dtype: int64
share Good or Great: 0.67
       books  minutes
count  12.00    12.00
mean    2.00    34.94
std     1.54    14.16
min     0.00    15.60
25%     1.00    24.45
50%     2.00    32.35
75%     3.00    43.70
max     5.00    60.20


**Answer.** Membership is nominal (student 5, public 4, staff 3) and rating is ordinal (Poor 1, OK 3, Good 5, Great 3), so 67% rated Good or Great. Books is discrete (mean 2.00) and minutes is continuous (mean 34.94, sd 14.16). Averages belong only to the last two columns.

## The same question in R, SPSS and Stata

**R**

```r
eye <- factor(c('brown','blue','brown','green','brown','blue','hazel','brown','green','blue'))  # Create eye as a factor, which is R's type for categories.
grade <- factor(c('Sophomore','Senior','Freshman','Junior','Sophomore','Senior','Freshman','Sophomore','Junior','Senior'),  # Create grade as a factor too. This line lists the ten grades, one per student.
  levels = c('Freshman','Sophomore','Junior','Senior'), ordered = TRUE)  # levels gives the order from Freshman to Senior, and ordered = TRUE tells R the categories are ranked.
siblings <- c(1, 0, 2, 3, 1, 1, 0, 2, 4, 1)  # siblings is a plain numeric vector, because it is a count.
height_cm <- c(172.4, 165.1, 180.3, 158.9, 175.6, 169.8, 162.2, 177.0, 171.3, 183.5)  # height_cm is a numeric vector too, because it is a measurement.
table(eye)  # table(eye) counts the students in each eye colour.
table(grade)  # table(grade) counts the students in each grade.
summary(siblings)  # summary(siblings) prints the minimum, quartiles, mean and maximum of a numeric vector.
summary(height_cm)  # summary(height_cm) does the same for the heights.
mean(eye)  # Deliberately average a category. R refuses by returning NA with a warning, showing the result is meaningless.
```

How to read it: table(eye) shows blue 3, brown 4, green 2, hazel 1, and table(grade) shows 2, 3, 2, 3. summary(siblings) gives a mean of 1.5 and summary(height_cm) a mean of 171.6. mean(eye) prints NA with a warning that the argument is not numeric or logical.

**SPSS** (Analyze > Descriptive Statistics > Frequencies (categories) and Descriptives (numbers))

```
DATA LIST FREE / eye (A6) grade siblings height.
BEGIN DATA
brown 2 1 172.4 blue 4 0 165.1 brown 1 2 180.3 green 3 3 158.9 brown 2 1 175.6
blue 4 1 169.8 hazel 1 0 162.2 brown 2 2 177.0 green 3 4 171.3 blue 4 1 183.5
END DATA.
VALUE LABELS grade 1 'Freshman' 2 'Sophomore' 3 'Junior' 4 'Senior'.
VARIABLE LEVEL eye (NOMINAL) /grade (ORDINAL) /siblings height (SCALE).
FREQUENCIES VARIABLES=eye grade.
DESCRIPTIVES VARIABLES=siblings height.
```

How to read it: The Frequencies tables show eye (blue 3, brown 4, green 2, hazel 1) and grade (Freshman 2, Sophomore 3, Junior 2, Senior 3) with percentages. The Descriptives table shows a Mean of 1.50 for siblings and 171.61 for height, with a height Std. Deviation of 7.88.

**Stata**

```
clear
set obs 10
generate eye = word("brown blue brown green brown blue hazel brown green blue", _n)
generate grade = real(word("2 4 1 3 2 4 1 2 3 4", _n))
generate siblings = real(word("1 0 2 3 1 1 0 2 4 1", _n))
generate height = real(word("172.4 165.1 180.3 158.9 175.6 169.8 162.2 177.0 171.3 183.5", _n))
label define g 1 "Freshman" 2 "Sophomore" 3 "Junior" 4 "Senior"
label values grade g
tabulate eye
tabulate grade
summarize siblings height
```

How to read it: tabulate eye gives blue 3, brown 4, green 2, hazel 1, and tabulate grade gives 2, 3, 2, 3 with percentages. summarize shows siblings with a mean of 1.5 and height with a mean of 171.61 and a standard deviation of 7.88.